
# <center> 🌐 🔤 <font color = 'lightblue'>**Data Science Concept Assistant**</font></center>


A two-iteration Retrieval-Augmented Generation (RAG) system built on 3 datasets.

| Iteration | Description |
|-----------|-------------|
| **1** | Manual FAISS semantic search (no LLM) |
| **2** | Full RAG pipeline with LangChain + Flan-T5 |

---

Datasets links:

*   [Machine Learning Questions](https://huggingface.co/datasets/mjphayes/machine_learning_questions)
*  [Data Science Q&A](https://huggingface.co/datasets/team-bay/data-science-qa://)
*   [Databricks Dolly 15k](https://huggingface.co/datasets/databricks/databricks-dolly-15k://)






---


In [ ]:
# Install all required dependencies
!pip install -q \
    datasets \
    sentence-transformers \
    faiss-cpu \
    langchain \
    langchain-core \
    langchain-community \
    langchain-huggingface \
    transformers \
    accelerate

---

## Iteration 1 — Manual FAISS Semantic Search

This iteration builds a lightweight semantic search engine from scratch using `sentence-transformers` and `FAISS`. No LLM is involved — retrieved documents are returned directly.

### 1.1 Load & Filter Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

print(f"Dataset size: {len(dataset)}")
print(f"\nSample entry:\n{dataset[0]}")



Dataset size: 15011

Sample entry:
{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [ ]:
# Keep only AI / Data Science entries in dolly-15
AI_KEYWORDS = [
    "machine learning", "data science", "neural network",
    "deep learning", "model", "training", "dataset",
    "statistics", "regression", "classification",
    "artificial intelligence"
]

def is_relevant(example: dict) -> bool:
    text = (example["instruction"] + " " + example["response"]).lower()
    return any(keyword in text for keyword in AI_KEYWORDS)

dataset = dataset.filter(is_relevant)
dataset = dataset.shuffle(seed=42).select(range(500)) #sample

print(f"Filtered dataset size: {len(dataset)}")
print(f"\nSample entry:\n{dataset[0]}")

Filtered dataset size: 500

Sample entry:
{'instruction': 'What is an operating model for AI?', 'context': '', 'response': 'It refers to the processes put in place to work with AI, from use case definition, to development to deployment and operation. Ultimately the objective of an operating model for AI is to streamline the value generation process from data to business results. Developing AI applications is a highly iterative process and generally strives in an agile environment.  However, organisations operate in a wide variety of contexts and come in many different shapes and sizes, hence it makes sense that there is no universal operating model for AI that fits everywhere. Regulatory requirements, data and resource availability and many other factors will play a role in determining the right operating model. \n\nIt is important then, to have a platform that is able to scale with the organisation as more use cases and users enter the pipeline, as more models are deployed and as more

### 1.2 Build Documents

In [ ]:
# Concatenate instruction + response into a single string per document
documents = [
    (item["instruction"] + " " + item["response"]).strip()
    for item in dataset
]

print(f"Total documents: {len(documents)}")
print(f"\nExample document:\n{documents[0][:300]}...")

Total documents: 500

Example document:
What is an operating model for AI? It refers to the processes put in place to work with AI, from use case definition, to development to deployment and operation. Ultimately the objective of an operating model for AI is to streamline the value generation process from data to business results. Develop...


### 1.3 Generate Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = encoder.encode(
    documents,
    batch_size=32,
    show_progress_bar=True
)

print(f"Embedding shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding shape: (500, 384)


### 1.4 Build FAISS Index

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print(f"Vectors indexed: {index.ntotal}")

Vectors indexed: 500


### 1.5 Semantic Search

In [ ]:
def semantic_search(query: str, k: int = 5) -> list[str]:
    """Return the top-k most relevant documents for a given query."""
    query_embedding = encoder.encode([query])
    _, indices = index.search(query_embedding, k)
    return [documents[i] for i in indices[0]]


# --- Test ---
query = "How do neural networks learn?"
results = semantic_search(query)

print(f"Query: {query}\n")
for i, result in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print(result[:400])
    print()

Query: How do neural networks learn?

--- Result 1 ---
What are artificial neural networks? Neural networks are computer systems inspired by the biological neural networks that constitute animal brains

--- Result 2 ---
Please give me a short bulleted list of the key components/architectures in a convolutional neural network. The key components/architecture of a convolutional neural network include:

Convolutional layers: These layers convolve the input matrix and reduce the number of free parameters and allows the network to be deeper.

Pooling layers: These layers reduce the dimensions of data by combining the 

--- Result 3 ---
given the following text, tell me the most popular method for training multi-layer perceptrons today and explain how it works Today, the most popular method for training multi-layer perceptrons (MLPs) is back-propagation. During backpropagation, the output values are compared with the correct answer to compute the value of some predefined error-function. The 

---

## Iteration 2 — Full RAG Pipeline with LangChain + Flan-T5

This iteration wraps the FAISS retriever inside a complete LangChain pipeline. Retrieved documents are formatted as few-shot examples and passed to **Flan-T5-Large** (encoder-decoder), which generates a final answer.

> **Model note:** `google/flan-t5-large` uses `text2text-generation` (encoder-decoder architecture). This is different from decoder-only models like Falcon or GPT, which use `text-generation`. Using the wrong task type is a common source of errors.

### 2.1 Load Dataset as LangChain Documents

In [ ]:
from datasets import load_dataset
from langchain_core.documents import Document

In [ ]:

# --- Load Dolly ---
ds_dolly = load_dataset("databricks/databricks-dolly-15k", split="train")

# --- Filter ---
AI_KEYWORDS = [
    "machine learning", "data science", "neural network",
    "deep learning", "model", "training", "dataset",
    "statistics", "regression", "classification",
    "artificial intelligence"
]

def is_relevant(example):
    text = (example["instruction"] + " " + example["response"]).lower()
    return any(keyword in text for keyword in AI_KEYWORDS)

ds_dolly_filtered = ds_dolly.filter(is_relevant)

# opcional: sample
ds_dolly_filtered = ds_dolly_filtered.shuffle(seed=42).select(range(500))

# --- Convert ---
def dolly_to_document(example):
    content = f"""Question: {example['instruction']}
Answer: {example['response']}"""
    return Document(
        page_content=content,
        metadata={"source": "dolly_filtered"}
    )

dolly_docs = [dolly_to_document(item) for item in ds_dolly_filtered]

print("Dolly filtrado:", len(dolly_docs))

Dolly filtrado: 500


In [ ]:
# --- Data Science QA ---
ds_data_science_qa = load_dataset("team-bay/data-science-qa", split="train")

def ds_qa_to_document(example):
    content = f"""Question: {example['question']}
Answer: {example['answer']}"""
    return Document(
        page_content=content,
        metadata={"source": "data_science_qa"}
    )

ds_qa_docs = [ds_qa_to_document(item) for item in ds_data_science_qa]


# --- ML Questions ---
ds_ml_questions = load_dataset("mjphayes/machine_learning_questions", split="train")

def ml_questions_to_document(example):
    content = f"""Question: {example['question']}
Answer: {example['answer']}"""
    return Document(
        page_content=content,
        metadata={"source": "ml_questions"}
    )

ml_qa_docs = [ml_questions_to_document(item) for item in ds_ml_questions]


print(
    f"DS QA: {len(ds_qa_docs)} | "
    f"ML QA: {len(ml_qa_docs)}"
)

DS QA: 473 | ML QA: 508


In [ ]:
all_docs = dolly_docs + ds_qa_docs + ml_qa_docs

print("Total docs:", len(all_docs))

Total docs: 1481


### 2.2 Build Vector Store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- Embeddings ---
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# --- Vector Store ---
vectorstore = FAISS.from_documents(all_docs, embedding_model)

# --- Retriever (MMR para diversidad entre datasets) ---
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,          # menos ruido, mejor para T5
        "fetch_k": 20,   # candidatos iniciales
        "lambda_mult": 0.6
    }
)

print(f"Vector store built with {len(all_docs)} documents. Retriever ready (MMR).")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store built with 1481 documents. Retriever ready (MMR).


verificar retrieval, expectativa: balance entre 3 datasets.

### 2.3 Prompt Template

In [ ]:
from langchain_core.prompts import PromptTemplate

PROMPT_TEMPLATE = """\
You are a data science assistant.

Answer the question in English, clearly and concisely (2-3 sentences).
Be precise and use technical terms when appropriate.
If the question is unrelated to data science, say "I don't know".

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=PROMPT_TEMPLATE
)

### 2.4 Load LLM — Flan-T5-Large

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_ID = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
flan_model.eval()


def generate_answer(prompt_text: str, max_new_tokens: int = 150) -> str:
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=512  # alineado con FLAN-T5
    )

    with torch.no_grad():
        output_ids = flan_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=False
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Formateo de texto

In [ ]:
def format_context(docs: list[Document]) -> str:
    chunks = []

    for doc in docs:
        text = doc.page_content.strip()

        # recorte controlado (evita overflow en T5)
        if len(text) > 400:
            text = text[:400]

        chunks.append(text)

    return "\n\n".join(chunks)

### 2.5 RAG Pipeline

In [ ]:
from langchain_core.documents import Document


def rag_pipeline(query: str) -> str:
    retrieved_docs = retriever.invoke(query)

    context = format_context(retrieved_docs)

    final_prompt = prompt.format(
        context=context,
        question=query
    )

    return generate_answer(final_prompt)

### 2.6 Test

In [ ]:
test_queries = [
    "Explain overfitting in machine learning.",
    "What is the difference between supervised and unsupervised learning?",
    "How does gradient descent work?"
]

for i, query in enumerate(test_queries, 1):
    try:
        answer = rag_pipeline(query)
    except Exception as e:
        answer = f"[ERROR] {str(e)}"

    print(f"\n--- Query {i} ---")
    print(f"Q: {query}")
    print(f"A: {answer}")


--- Query 1 ---
Q: Explain overfitting in machine learning.
A: Overfitting is when a model learns the training data too well, including its peculiarities, and fails to generalize to new, unseen data.

--- Query 2 ---
Q: What is the difference between supervised and unsupervised learning?
A: Supervised learning involves predicting an output or response variable based on input features, with known output values in the training data. Unsupervised learning, in contrast, focuses on understanding the structure of data without predefined output variables, often to find patterns or groupings within the data.

--- Query 3 ---
Q: How does gradient descent work?
A: Gradient Descent is an optimization algorithm used to minimize the loss function or error of a model by iteratively updating the model parameters in the direction of the steepest descent of the gradient.


In [ ]:
def debug_retrieval(query: str, max_chars: int = 300):
    docs = retriever.invoke(query)

    print(f"\nQuery: {query}")
    print(f"Retrieved documents: {len(docs)}")

    for i, d in enumerate(docs, 1):
        content = d.page_content.strip()
        preview = content[:max_chars]

        print(f"\n--- Doc {i} ---")
        print(preview)
        print(f"[length={len(content)}]")
        print(f"source={d.metadata.get('source', 'unknown')}")

In [ ]:
debug_retrieval("What is overfitting in machine learning?")


Query: What is overfitting in machine learning?
Retrieved documents: 3

--- Doc 1 ---
Question: What is overfitting in machine learning?
Answer: Overfitting is when a model learns the training data too well, including its peculiarities, and fails to generalize to new, unseen data.
[length=195]
source=ml_questions

--- Doc 2 ---
Question: How can overfitting be addressed in machine learning?
Answer: Overfitting can be addressed by using more training data and regularization, which penalizes models for being overly complex.
[length=197]
source=ml_questions

--- Doc 3 ---
Question: What is 'overfitting' and how can it be prevented?
Answer: Overfitting occurs when a machine learning model learns the training data too well, including its noise and details, reducing its performance on new data. It can be prevented by techniques like cross-validation, regularization, and
[length=327]
source=ml_questions


In [ ]:
docs = retriever.invoke(query)

for i, d in enumerate(docs):
    print(f"\n--- Doc {i+1} ---")
    print(d.page_content[:300])
    print(d.metadata)


--- Doc 1 ---
Question: what is Gradient Descent
Answer: Gradient Descent is an optimization algorithm used to minimize the loss function or error of a model by iteratively updating the model parameters in the direction of the steepest descent of the gradient, enabling efficient training of machine learning model
{'source': 'data_science_qa'}

--- Doc 2 ---
Question: please explain Gradient Vector
Answer: The Gradient Vector is a vector of partial derivatives representing the rate of change of a scalar-valued function with respect to each independent variable in multivariable calculus, providing directional information and pointing towards the directio
{'source': 'data_science_qa'}

--- Doc 3 ---
Question: What is the role of gradient descent in linear regression?
Answer: Gradient descent is used to find the minimum of the loss function by iteratively adjusting the model parameters.
{'source': 'ml_questions'}


Gradio UI / UX

In [ ]:
!pip install gradio

Función para respuesta y contexto

In [ ]:
def rag_pipeline_with_context(query: str, k: int = 4):
    retrieved_docs = retriever.invoke(query)

    # limitar chunks (mejora UX)
    retrieved_docs = retrieved_docs[:k]

    context = format_context(retrieved_docs)

    final_prompt = prompt.format(
        context=context,
        question=query
    )

    answer = generate_answer(final_prompt)

    return {
        "answer": answer,
        "context": [doc.page_content for doc in retrieved_docs]
    }

---

# Inference & Qualitative Evaluation of Outputs

In [ ]:
test_set = [
    {"question": "What is overfitting?",
     "expected": "Overfitting is when a model learns noise instead of general patterns."},

    {"question": "What is underfitting?",
     "expected": "Underfitting is when a model is too simple to capture the underlying pattern."},

    {"question": "What is a neural network?",
     "expected": "A neural network is a model inspired by the structure of the brain."},

    {"question": "What is supervised learning?",
     "expected": "Supervised learning uses labeled data to learn a mapping from inputs to outputs."},

    {"question": "What is unsupervised learning?",
     "expected": "Unsupervised learning finds patterns in data without labeled outputs."},

    {"question": "What is a training set?",
     "expected": "A training set is the data used to train a model."},

    {"question": "What is a test set?",
     "expected": "A test set is used to evaluate a model's performance on unseen data."},

    {"question": "What is validation data?",
     "expected": "Validation data is used to tune model parameters during training."},

    {"question": "What is bias in machine learning?",
     "expected": "Bias is error introduced by overly simple assumptions in the model."},

    {"question": "What is variance in machine learning?",
     "expected": "Variance is how much a model's predictions change with different data."},

    {"question": "What is the bias-variance tradeoff?",
     "expected": "It is the balance between model simplicity and sensitivity to data."},

    {"question": "What is a feature?",
     "expected": "A feature is an input variable used by a model."},

    {"question": "What is a label?",
     "expected": "A label is the target output the model is trying to predict."},

    {"question": "What is regression?",
     "expected": "Regression predicts continuous numerical values."},

    {"question": "What is classification?",
     "expected": "Classification predicts discrete categories or classes."},

    {"question": "What is a loss function?",
     "expected": "A loss function measures how wrong a model's predictions are."},

    {"question": "What is gradient descent?",
     "expected": "Gradient descent is an optimization method to minimize loss."},

    {"question": "What is a learning rate?",
     "expected": "The learning rate controls how large the update steps are during training."},

    {"question": "What is normalization?",
     "expected": "Normalization rescales features to a common range."},

    {"question": "What is standardization?",
     "expected": "Standardization transforms data to have zero mean and unit variance."},

    {"question": "What is cross-validation?",
     "expected": "Cross-validation evaluates a model using multiple data splits."},

    {"question": "What is a confusion matrix?",
     "expected": "A confusion matrix shows prediction results against actual labels."},

    {"question": "What is precision?",
     "expected": "Precision is the fraction of correct positive predictions."},

    {"question": "What is recall?",
     "expected": "Recall is the fraction of actual positives correctly identified."},

    {"question": "What is F1 score?",
     "expected": "F1 score is the harmonic mean of precision and recall."},

    {"question": "What is clustering?",
     "expected": "Clustering groups similar data points without labels."},

    {"question": "What is k-means?",
     "expected": "K-means is a clustering algorithm that partitions data into k groups."},

    {"question": "What is dimensionality reduction?",
     "expected": "Dimensionality reduction reduces the number of features while preserving information."},

    {"question": "What is PCA?",
     "expected": "PCA is a technique that projects data onto principal components."},

    {"question": "What is regularization?",
     "expected": "Regularization adds constraints to prevent overfitting."},

    {"question": "¿Qué es la inteligencia artificial?",
     "expected": "Artificial intelligence is the simulation of human intelligence by machines."}
]

In [ ]:
out_of_domain_test_set = [
    {"question": "How do you make coffee?",
     "expected": "I don't know"},

    {"question": "What is the capital of France?",
     "expected": "I don't know"},

    {"question": "Who won the last World Cup?",
     "expected": "I don't know"},

    {"question": "How do I lose weight fast?",
     "expected": "I don't know"},

    {"question": "What is the best movie of all time?",
     "expected": "I don't know"},

    {"question": "How does a car engine work?",
     "expected": "I don't know"},

    {"question": "Write a short poem about love.",
     "expected": "I don't know"},

    {"question": "What is photosynthesis?",
     "expected": "I don't know"},

    {"question": "How do I fix a leaking faucet?",
     "expected": "I don't know"},

    {"question": "What is the meaning of life?",
     "expected": "I don't know"},

    {"question": "¿Cómo es la receta de una torta ángel?",
     "expected": "I don't know"}
]

In [ ]:
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


# combinar datasets
combined_test_set = []

for item in test_set:
    combined_test_set.append({
        "question": item["question"],
        "expected": item["expected"],
        "type": "in_domain"
    })

for item in out_of_domain_test_set:
    combined_test_set.append({
        "question": item["question"],
        "expected": item["expected"],
        "type": "out_of_domain"
    })


results = []

for item in combined_test_set:
    result = rag_pipeline_with_context(item["question"])
    pred = result["answer"]
    score = similarity(pred, item["expected"])

    results.append({
        "question": item["question"],
        "prediction": pred,
        "expected": item["expected"],
        "score": score,
        "type": item["type"]
    })


# métricas separadas
in_domain_scores = [r["score"] for r in results if r["type"] == "in_domain"]
out_domain_scores = [r["score"] for r in results if r["type"] == "out_of_domain"]

print("In-domain avg:", round(sum(in_domain_scores)/len(in_domain_scores), 2))
print("Out-of-domain avg:", round(sum(out_domain_scores)/len(out_domain_scores), 2))

In-domain avg: 0.36
Out-of-domain avg: 0.9


In [ ]:
for r in results[:5]:
    print(r["type"], "| Q:", r["question"])
    print("Pred:", r["prediction"])
    print("Score:", round(r["score"], 2))
    print("-" * 40)

in_domain | Q: What is overfitting?
Pred: Overfitting occurs when a machine learning model learns the training data too well, including its noise and details, reducing its performance on new data.
Score: 0.47
----------------------------------------
in_domain | Q: What is underfitting?
Pred: Underfitting occurs when a model is too simple to capture the underlying structure of the data, resulting in poor performance on both training and test data.
Score: 0.64
----------------------------------------
in_domain | Q: What is a neural network?
Pred: A neural network is a computational model inspired by the structure and function of the human brain, consisting of interconnected nodes (neurons) organized in layers.
Score: 0.58
----------------------------------------
in_domain | Q: What is supervised learning?
Pred: Supervised learning is a machine learning paradigm where the algorithm learns from labeled data, with each example paired with a desired output.
Score: 0.48
----------------------

In [ ]:
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template='You are a data science assistant.\n\nAnswer the question in English, clearly and concisely (2-3 sentences).\nBe precise and use technical terms when appropriate.\nIf the question is unrelated to data science, say "I don\'t know".\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:'


In [ ]:
results = rag_pipeline_with_context(item["question"])

pred = results["answer"]
retrieved_docs = results["context"]

print(retrieved_docs[:2])

['Question: Who is Paulina Flores Arias\nAnswer: Paulina Flores Arias (born 1980 in Culiacán, Sinaloa) is a Mexican model, who after winning the national title of Nuestra Belleza México, represented her country in the 2000 Miss World pageant, held in London, England, on November 30, 2000. Paulina is currently a professional fashion model, and has been represented by several national and international modeling agencies.', 'Question: Using Max Weber\'s ideal bureaucracy as outlined in the provided text, generate 5 pro and 5 con bullet points on how bureaucracy is applied to an organization in practice.\nAnswer: Pro:\n1. References are provided and checked to ensure candidates are qualified.\n2. Management priorities are completed by employees as soon as possible.\n3. "Industry experts" are welcome.\n4. Learning is rewarded and encouraged. Free training is provided. \n5. When warranted, a RACI (responsible,  accountable, consulted, and informed) is shared across departments. \n\nCon: \n1.

In [ ]:
print(docs)

[Document(id='2c790003-4229-4b09-a60c-4e87d4963a6d', metadata={'source': 'data_science_qa'}, page_content='Question: what is Gradient Descent\nAnswer: Gradient Descent is an optimization algorithm used to minimize the loss function or error of a model by iteratively updating the model parameters in the direction of the steepest descent of the gradient, enabling efficient training of machine learning models through backpropagation and parameter updates.'), Document(id='b71a5690-8545-4fd6-a558-079fb07429a7', metadata={'source': 'data_science_qa'}, page_content='Question: please explain Gradient Vector\nAnswer: The Gradient Vector is a vector of partial derivatives representing the rate of change of a scalar-valued function with respect to each independent variable in multivariable calculus, providing directional information and pointing towards the direction of steepest ascent of the function.'), Document(id='0e49b089-3823-4357-bb80-9e8e2cb580a4', metadata={'source': 'ml_questions'}, pag

**Final results are looking good**. The out-of-domain avg means it's working quite fine for its limitatiions. Meanwhile, the In-domain avg is still down, but considering that the SequenceMatcher doesn't measure the exact overall semantic meaning of the text, it's ok.

---

# Demo on Gradio

Wrapper for Gradio

In [ ]:
def gradio_fn(query, show_context):
    result = rag_pipeline_with_context(query)

    context_text = ""
    if show_context:
        context_text = "\n\n---\n\n".join(result["context"])

    return result["answer"], context_text

Gradio UI Demo

In [ ]:
import gradio as gr

with gr.Blocks(
    theme=gr.Theme.from_hub("davehornik/Tealy"),
    css="""
    .custom-label label {
        color: #14b8a6;
        font-weight: bold;
    }
    .gradio-container {
        background: #0f0f0f;
    }

    #ask-btn {
        background: #14b8a6;
        color: #2E2A24;
        border: none;
        border-radius: 8px;
    }

    #ask-btn:hover {
        background: #ffffff;
    }
    """
) as demo:
    gr.Markdown("""
<h1 style="
    text-align:center;
    color:#008080;
    background:#dadbdb;
    padding:16px;
    border-radius:12px;
">
🌐 Data Science >_ One Concept at a Time 🌐
</h1>
""")
    gr.Markdown("Simple questions, simple answers. Toggle context if you want the behind-the-scenes.")

    with gr.Row():
        query = gr.Textbox(
            label="Question",
            placeholder="e.g. What is overfitting?",
            lines=2,
            elem_classes="custom-label"
        )

    show_context = gr.Checkbox(label="Show retrieved context", value=False)

    submit = gr.Button("Ask", elem_id="ask-btn")

    answer_output = gr.Textbox(label="Answer", lines=4, elem_classes="custom-label")
    context_output = gr.Textbox(label="Retrieved context", lines=10, elem_classes="custom-label")

    submit.click(
        fn=gradio_fn,
        inputs=[query, show_context],
        outputs=[answer_output, context_output]
    )

demo.launch()

/tmp/ipykernel_9448/1133978890.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_9448/1133978890.py:3: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0af7985624cc576e41.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
